# 数据库连接测试

本 Notebook 测试 FastGPT 环境中各个数据库组件的连接。

## 1. MongoDB 连接测试

In [ ]:
from pymongo import MongoClient
import json

try:
    # 连接 MongoDB
    client = MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=5000)
    
    # 获取服务器信息
    server_info = client.server_info()
    print("✓ MongoDB 连接成功")
    print(f"  版本: {server_info.get('version', 'N/A')}")
    print(f"  Git 版本: {server_info.get('gitVersion', 'N/A')}")
    
    # 列出所有数据库
    databases = client.list_database_names()
    print(f"\n数据库列表: {databases}")
    
except Exception as e:
    print(f"✗ MongoDB 连接失败: {e}")

## 2. PostgreSQL 连接测试

In [ ]:
import psycopg2
from psycopg2 import sql

try:
    # 连接 PostgreSQL
    conn = psycopg2.connect(
        host='localhost',
        port=5432,
        database='postgres',
        user='postgres',
        password='aiproxy'
    )
    
    print("✓ PostgreSQL 连接成功")
    
    # 创建游标并执行查询
    cur = conn.cursor()
    
    # 获取 PostgreSQL 版本
    cur.execute("SELECT version();")
    db_version = cur.fetchone()
    print(f"  版本: {db_version[0]}")
    
    # 列出所有数据库
    cur.execute("SELECT datname FROM pg_database WHERE datistemplate = false;")
    databases = cur.fetchall()
    print(f"\n数据库列表:")
    for db in databases:
        print(f"  - {db[0]}")
    
    # 检查 pgvector 扩展
    cur.execute("SELECT extname FROM pg_extension WHERE extname = 'vector';")
    vector_ext = cur.fetchone()
    if vector_ext:
        print("\n✓ pgvector 扩展已安装")
    else:
        print("\n✗ pgvector 扩展未安装")
    
    cur.close()
    conn.close()
    
except Exception as e:
    print(f"✗ PostgreSQL 连接失败: {e}")

## 3. Redis 连接测试

In [ ]:
import redis

try:
    # 连接 Redis
    r = redis.Redis(
        host='localhost',
        port=6379,
        password='mypassword',
        decode_responses=True
    )
    
    # 测试连接
    r.ping()
    print("✓ Redis 连接成功")
    
    # 获取服务器信息
    info = r.info('server')
    print(f"  版本: {info.get('redis_version', 'N/A')}")
    print(f"  运行模式: {info.get('redis_mode', 'N/A')}")
    
    # 测试读写操作
    r.set('test_key', 'Hello from FastGPT on Binder!')
    value = r.get('test_key')
    print(f"\n  测试写入: test_key = '{value}'")
    
    # 清理测试数据
    r.delete('test_key')
    print("  ✓ 测试数据已清理")
    
except Exception as e:
    print(f"✗ Redis 连接失败: {e}")

## 4. MinIO 连接测试

In [ ]:
from minio import Minio
from minio.error import S3Error

try:
    # 创建 MinIO 客户端
    client = Minio(
        'localhost:9000',
        access_key='minioadmin',
        secret_key='minioadmin',
        secure=False
    )
    
    print("✓ MinIO 连接成功")
    
    # 列出所有存储桶
    buckets = client.list_buckets()
    print(f"\n存储桶列表:")
    for bucket in buckets:
        print(f"  - {bucket.name} (创建时间: {bucket.creation_date})")
    
    # 如果没有存储桶，创建一个测试桶
    if not buckets:
        test_bucket = 'test-binder-bucket'
        client.make_bucket(test_bucket)
        print(f"\n✓ 创建测试存储桶: {test_bucket}")
        
        # 删除测试桶
        client.remove_bucket(test_bucket)
        print(f"✓ 删除测试存储桶: {test_bucket}")
    
except S3Error as e:
    print(f"✗ MinIO 连接失败: {e}")
except Exception as e:
    print(f"✗ MinIO 连接失败: {e}")

## 5. ChromaDB 测试

In [ ]:
import chromadb

try:
    # 创建 ChromaDB 客户端
    client = chromadb.Client()
    
    print("✓ ChromaDB 初始化成功")
    
    # 创建或获取集合
    collection = client.get_or_create_collection(name="test-collection")
    print(f"  集合名称: {collection.name}")
    print(f"  文档数量: {collection.count()}")
    
    # 添加测试文档
    collection.add(
        documents=["This is a test document"],
        metadatas=[{"source": "binder-test"}],
        ids=["id1"]
    )
    print(f"\n✓ 添加测试文档成功")
    print(f"  当前文档数量: {collection.count()}")
    
    # 查询文档
    results = collection.query(
        query_texts=["test"],
        n_results=1
    )
    print(f"\n✓ 查询测试成功")
    print(f"  查询结果: {results['documents']}")
    
    # 清理测试数据
    client.delete_collection("test-collection")
    print("\n✓ 测试数据已清理")
    
except Exception as e:
    print(f"✗ ChromaDB 测试失败: {e}")

## ✅ 测试总结

如果所有数据库连接测试都通过，说明你的 FastGPT 数据层已正确配置！

**下一步**:
- 尝试在 `03-api-testing.ipynb` 中测试 FastGPT API
- 查看 [QUICKSTART_BINDER.md](../QUICKSTART_BINDER.md) 了解更多使用技巧